# Seminar Notebook 3.6: Classifying Names by Gender

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook applies the classification techniques from notebook 3.4 to a different problem: predicting gender from first names using character-level features. Instead of words in documents, our "features" are characters (and character n-grams) in names. All the same principles apply.

## Set up steps

### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
if not os.path.exists(sdir):
    os.makedirs(sdir)

### Getting and loading the data

The file `EN-names.csv` is available on the course GitHub site and it contains a list of nearly 25,000 popular names in the U.S., labelled by the most frequent gender based on Social Security records. You can get more details here: <https://catalog.data.gov/dataset/baby-names-from-social-security-card-applications-national-data>.

In [ ]:
import os
import requests

rfile = "https://raw.githubusercontent.com/lse-my459/data/master/EN-names.csv"
lfile = os.path.join(sdir, os.path.basename(rfile))

if not os.path.exists(lfile):
    r = requests.get(rfile)
    r.raise_for_status()
    with open(lfile, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {lfile}")
else:
    print(f"File already exists: {lfile}")

Now, we load the data as `nf` (for "names dataframe") and preview it.

In [ ]:
import pandas as pd

# Load the names dataset
nf = pd.read_csv(lfile)

print("First five rows of the dataframe:")
print(nf.head())
print()
print("Gender break-down of names:")
print(nf["gender"].value_counts())

### Creating a character-level DFM

Instead of tokenising documents into words, we tokenise names into individual characters. Each name is a "document" and each character is a "feature." We use sklearn's `CountVectorizer` with `analyzer="char"` to do this.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create a character-level DFM (unigrams only)
vec = CountVectorizer(analyzer="char", lowercase=True)
dfm = vec.fit_transform(nf["name"])
vocabulary = vec.get_feature_names_out()

print(dfm.shape)
print(list(vocabulary))

### Labelling and splitting

We create binary labels (1 = female, 0 = male) and split into 80% training and 20% test. Since we will create multiple DFMs with different feature sets, we split by index so that the same names end up in the training and test sets regardless of which DFM we use.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Create binary labels
labels = (nf["gender"] == "female").astype(int).values

# Split indices (not the data itself)
indices = np.arange(len(labels))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=9564)

# Split unigram DFM and labels
dfm_train = dfm[train_idx]
dfm_test = dfm[test_idx]
labels_train = labels[train_idx]
labels_test = labels[test_idx]

print(f"Training set: {len(train_idx)} names")
print(f"Test set: {len(test_idx)} names")
print(f"Training labels: {pd.Series(labels_train).value_counts().to_dict()}")
print(f"Test labels: {pd.Series(labels_test).value_counts().to_dict()}")

## Classifying with Naive Bayes (character unigrams)

Let's start with a Multinomial Naive Bayes classifier using individual characters as features. Can the distribution of characters in a name predict whether it belongs to a male or female?

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Train Naive Bayes
nb = MultinomialNB(alpha=1.0)
nb.fit(dfm_train, labels_train)

# Predict
nb_preds_test = nb.predict(dfm_test)

# Confusion matrix
nb_cm = pd.crosstab(labels_test, nb_preds_test, rownames=["True"], colnames=["Predicted"])
print("Confusion matrix:")
print(nb_cm)
print()
print(classification_report(labels_test, nb_preds_test, target_names=["male", "female"]))

This is pretty good, but with only 26 or so character features (unigrams), there isn't much information to work with. Let's look at which characters are most predictive by examining the estimated word probabilities for each class.

In [ ]:
# Log probabilities for each feature given class
# nb.feature_log_prob_ has shape (n_classes, n_features)
log_probs = pd.DataFrame(
    nb.feature_log_prob_,
    columns=vocabulary,
    index=["male", "female"]
)

# Which characters are most "female" vs "male"?
diff = log_probs.loc["female"] - log_probs.loc["male"]
diff = diff.sort_values()

print("Most 'male' characters (lowest log-prob difference):")
print(diff.head(5).round(3))
print()
print("Most 'female' characters (highest log-prob difference):")
print(diff.tail(5).round(3))

## Improving with character n-grams

Character unigrams lose all positional information. The letter "a" at the end of a name (e.g., "Maria") is much more predictive of gender than "a" in the middle (e.g., "James"). Character n-grams capture some of this structure. Let's rebuild the DFM using character bigrams and trigrams alongside unigrams.

In [ ]:
# Character n-grams (1 to 3)
vec_ngram = CountVectorizer(analyzer="char", ngram_range=(1, 3), lowercase=True, min_df=20)
dfm_ngram = vec_ngram.fit_transform(nf["name"])
vocab_ngram = vec_ngram.get_feature_names_out()

print(dfm_ngram.shape)

In [ ]:
# Split the n-gram DFM using the same indices
dfm_ng_train = dfm_ngram[train_idx]
dfm_ng_test = dfm_ngram[test_idx]

# Train Naive Bayes on n-grams
nb_ng = MultinomialNB(alpha=1.0)
nb_ng.fit(dfm_ng_train, labels_train)

# Predict
nb_ng_preds = nb_ng.predict(dfm_ng_test)

# Results
nb_ng_cm = pd.crosstab(labels_test, nb_ng_preds, rownames=["True"], colnames=["Predicted"])
print("Confusion matrix (character n-grams):")
print(nb_ng_cm)
print()
print(classification_report(labels_test, nb_ng_preds, target_names=["male", "female"]))

Character n-grams improves performance because they capture patterns like names ending in "a", "na", "ina" (often female) vs. names ending in "on", "an", "ck" (often male).

## Classifying with ridge regression

Let's try regularised regression on the n-gram features. As in notebook 3.4, we apply TF-IDF weighting first.

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

# TF-IDF
tfidf = TfidfTransformer()
dfm_ng_train_tfidf = tfidf.fit_transform(dfm_ng_train)
dfm_ng_test_tfidf = tfidf.transform(dfm_ng_test)

# Cross-validation for lambda
cv = KFold(n_splits=5, shuffle=True, random_state=459)
lambdas = np.logspace(-4, 4, 20)

# Fit ridge
ridge = RidgeCV(alphas=lambdas, cv=cv)
ridge.fit(dfm_ng_train_tfidf, labels_train)

print(f"Best lambda: {ridge.alpha_:.4f}")

In [ ]:
# Predict and evaluate
ridge_scores = ridge.predict(dfm_ng_test_tfidf)
ridge_preds = (ridge_scores >= 0.5).astype(int)

ridge_cm = pd.crosstab(labels_test, ridge_preds, rownames=["True"], colnames=["Predicted"])
print("Confusion matrix (ridge):")
print(ridge_cm)
print()
print(classification_report(labels_test, ridge_preds, target_names=["male", "female"]))

### Most predictive n-grams

With ridge regression, we can examine the coefficients to see which character patterns are most predictive of each gender.

In [ ]:
coefs = pd.Series(ridge.coef_, index=vocab_ngram)

print("Most 'male' n-grams (most negative coefficients):")
print(coefs.sort_values().head(10).round(4))
print()
print("Most 'female' n-grams (most positive coefficients):")
print(coefs.sort_values().tail(10).round(4))

## Comparing classifiers

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_metrics(true, pred, name):
    """Calculate performance metrics for the positive class (female = 1)."""
    return {
        "Classifier": name,
        "Accuracy": accuracy_score(true, pred),
        "Precision": precision_score(true, pred),
        "Recall": recall_score(true, pred),
        "F1": f1_score(true, pred)
    }

comparison = pd.DataFrame([
    get_metrics(labels_test, nb_preds_test, "NB (unigrams)"),
    get_metrics(labels_test, nb_ng_preds, "NB (n-grams)"),
    get_metrics(labels_test, ridge_preds, "Ridge (n-grams)"),
]).set_index("Classifier").round(3)

comparison

The comparison shows two things: (1) character n-grams are much more informative than character unigrams for this task, and (2) the choice of classifier (Naive Bayes vs. ridge) also matters, though less than the choice of features.

## Try it yourself

Pick a name and see what the classifiers predict!

In [ ]:
test_names = ["Ryan", "Maria", "Alex", "Jordan", "Priya", "Kenji"]

# Transform using the n-gram vectorizer
test_dfm = vec_ngram.transform(test_names)
test_tfidf = tfidf.transform(test_dfm)

# Predictions from each classifier
nb_ng_test_preds = nb_ng.predict(test_dfm)
ridge_test_scores = ridge.predict(test_tfidf)
ridge_test_preds = (ridge_test_scores >= 0.5).astype(int)

results = pd.DataFrame({
    "Name": test_names,
    "NB prediction": ["female" if p == 1 else "male" for p in nb_ng_test_preds],
    "Ridge prediction": ["female" if p == 1 else "male" for p in ridge_test_preds],
    "Ridge score": ridge_test_scores.round(3)
})
results